In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import re
from langdetect import detect

df = pd.read_excel("/content/总reviews1.xlsx")


df['review_text'] = df['review_text'].fillna("").astype(str)
df['title'] = df['title'].fillna("").astype(str)
df['place_title'] = df['place_title'].fillna("").astype(str)
df['user_name'] = df['user_name'].fillna("").astype(str)
-

resident_keywords = [
    "always come", "been going", "local", "from around here",
    "long time", "regular", "our usual place",
    "since childhood", "neighborhood", "nearby",
    "favorite pub", "know the owner",
    "every week", "every day", "after work",
    "lunch break", "my usual place"
]

tourist_keywords = [
    "first time", "trip", "holiday", "tour", "visit",
    "sightseeing", "bucket list", "guide", "itinerary",
    "instagrammable", "must visit", "amazing view",
    "stunning", "photo spot", "as a tourist"
]


resident_patterns = [
    r"\b(i have been|been going|coming here) for \d+ years",
    r"\balways come\b",
    r"\blocal\b",
    r"\bregular\b",
]

tourist_patterns = [
    r"\btravel\b",
    r"\bon vacation\b",
    r"\btrip\b",
    r"\bitinerary\b",
    r"\bvisited\b",
    r"\bholiday\b",
]


tourist_place_types = [
    "church", "cathedral", "museum", "palace", "bridge",
    "attraction", "square", "monument", "landmark",
    "tower", "gallery"
]

resident_place_types = [
    "supermarket", "pharmacy", "market", "bakery",
    "cafe", "restaurant", "bar", "shop"
]


tourist_months = [4, 6, 7, 8, 12]



def classify_final(row):

    text = (row["review_text"] + " " + row["title"]).lower()
    place = row["place_title"].lower()



    try:
        if detect(text) != "en":
            return "Tourist"
    except:
        pass



    if any(k in text for k in resident_keywords):
        return "Resident"

    if any(k in text for k in tourist_keywords):
        return "Tourist"

    # ---------------------
    if any(re.search(p, text) for p in resident_patterns):
        return "Resident"

    if any(re.search(p, text) for p in tourist_patterns):
        return "Tourist"

    # ---------------------
    if any(w in place for w in tourist_place_types):
        return "Tourist"

    if any(w in place for w in resident_place_types):
        return "Resident"

    # ---------------------
    if str(row.get("images", "")).count("http") >= 3:
        return "Tourist"

    # ---------------------
    if df[df["user_name"] == row["user_name"]].shape[0] == 1:
        return "Tourist"

    # ---------------------
    try:
        month = pd.to_datetime(row["date"]).month
        if month in tourist_months:
            return "Tourist"
    except:
        pass

    # ---------------------
    if len(text.split()) > 80:
        return "Resident"

    # ---------------------
    return "Tourist"


# -----------------------------------------------------------
df["final_label"] = df.apply(classify_final, axis=1)


# -----------------------------------------------------------
df.to_csv("final_classified_reviews.csv", index=False)

print("final_classified_reviews.csv")

final_classified_reviews.csv


In [ ]:
import pandas as pd
import json


df = pd.read_csv("/content/final_classified_reviews.csv")

lat_col = "lat_place"
lon_col = "lng_place"


df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')

df = df.dropna(subset=[lat_col, lon_col])


# ===============================
# ===============================
tourist_df = df[df["final_label"] == "Tourist"]
resident_df = df[df["final_label"] == "Resident"]


# ===============================
# ===============================
tourist_df.to_csv("tourist_reviews.csv", index=False)
resident_df.to_csv("resident_reviews.csv", index=False)

print("✅ CSV 已导出")


# ===============================
# ===============================
def df_to_geojson(df, lat_col, lon_col):

    features = []

    for _, row in df.iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [
                    float(row[lon_col]),
                    float(row[lat_col])
                ]
            },
            "properties": row.drop([lat_col, lon_col]).to_dict()
        }
        features.append(feature)

    geojson = {
        "type": "FeatureCollection",
        "features": features
    }

    return geojson


# ===============================
# ===============================
tourist_geojson = df_to_geojson(tourist_df, lat_col, lon_col)
resident_geojson = df_to_geojson(resident_df, lat_col, lon_col)

with open("tourist_reviews.geojson", "w", encoding="utf-8") as f:
    json.dump(tourist_geojson, f, ensure_ascii=False, indent=2)

with open("resident_reviews.geojson", "w", encoding="utf-8") as f:
    json.dump(resident_geojson, f, ensure_ascii=False, indent=2)

print("✅ GeoJSON 已导出")

✅ CSV 已导出
✅ GeoJSON 已导出
